In [11]:
import os
import pandas as pd
from sqlalchemy import create_engine

DB_PATH = "bluestock_mf.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

os.makedirs("data/processed", exist_ok=True)

NAV_HISTORY_SOURCE = r"C:\Users\dixit\bluestock_mf_capstone\data\raw\02_nav_history.csv"
PERFORMANCE_SOURCE = r"C:\Users\dixit\bluestock_mf_capstone\data\raw\07_scheme_performance.csv"
TRANSACTIONS_SOURCE = r"C:\Users\dixit\bluestock_mf_capstone\data\raw\08_investor_transactions.csv"

def clean_nav_history():
    print(f"Reading from: {NAV_HISTORY_SOURCE}")
    df = pd.read_csv(NAV_HISTORY_SOURCE)
    df.columns = df.columns.str.strip()
    
    df['date'] = pd.to_datetime(df['date'])
    
    df = df.drop_duplicates(subset=['amfi_code', 'date'])
    
    df = df.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)
    
    df['nav'] = df.groupby('amfi_code')['nav'].ffill()
    
    df = df[df['nav'] > 0]
    
    df.to_csv("data/processed/nav_history_clean.csv", index=False)
    df.to_sql("fact_nav", engine, if_exists="replace", index=False)
    print(f" Loaded fact_nav: {len(df)} rows\n")


def clean_transactions():
    print(f"Reading from: {TRANSACTIONS_SOURCE}")
    df = pd.read_csv(TRANSACTIONS_SOURCE)
    df.columns = df.columns.str.strip()
    
    if 'amount_inr' in df.columns:
        df = df.rename(columns={'amount_inr': 'amount'})
        
    type_mapping = {
        'sip': 'SIP', 'SIP': 'SIP', 'Systematic Investment Plan': 'SIP',
        'lumpsum': 'Lumpsum', 'Lumpsum': 'Lumpsum', 'Single': 'Lumpsum',
        'redemption': 'Redemption', 'Redemptions': 'Redemption', 'Withdrawal': 'Redemption'
    }
    df['transaction_type'] = df['transaction_type'].map(type_mapping).fillna('Other')
    
    df = df[df['amount'] > 0]
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    valid_kyc = ['Verified', 'Pending', 'Failed']
    df['kyc_status'] = df['kyc_status'].apply(lambda x: x if x in valid_kyc else 'Pending')
    
    df.to_csv("data/processed/investor_transactions_clean.csv", index=False)
    df.to_sql("fact_transactions", engine, if_exists="replace", index=False)
    print(f" Loaded fact_transactions: {len(df)} rows\n")


def clean_performance():
    print(f"Reading from: {PERFORMANCE_SOURCE}")
    df = pd.read_csv(PERFORMANCE_SOURCE)
    df.columns = df.columns.str.strip()
    
    rename_rules = {
        'return_1yr_pct': 'return_1y',
        'return_3yr_pct': 'return_3y',
        'return_5yr_pct': 'return_5y',
        'expense_ratio_pct': 'expense_ratio'
    }
    df = df.rename(columns=rename_rules)
    
    return_cols = ['return_1y', 'return_3y', 'return_5y']
    for col in return_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    df['is_anomaly'] = ((df['return_1y'] > 100) | (df['return_1y'] < -50)).astype(int)
    
    df['expense_ratio'] = pd.to_numeric(df['expense_ratio'], errors='coerce')
    df['expense_ratio'] = df['expense_ratio'].clip(0.1, 2.5)
    
    if 'aum_crore' in df.columns:
        aum_df = df[['amfi_code', 'aum_crore']].dropna().copy()
        aum_df = aum_df.rename(columns={'aum_crore': 'aum_amount'})
        aum_df['as_of_date'] = "2026-06-03"
        aum_df.to_sql("fact_aum", engine, if_exists="replace", index=False)
        print(f" Loaded fact_aum: {len(aum_df)} rows")

    final_cols = ['amfi_code', 'return_1y', 'return_3y', 'return_5y', 'expense_ratio', 'is_anomaly']
    df_performance_final = df[final_cols]
    
    df_performance_final.to_csv("data/processed/scheme_performance_clean.csv", index=False)
    df_performance_final.to_sql("fact_performance", engine, if_exists="replace", index=False)
    print(f" Loaded fact_performance: {len(df_performance_final)} rows\n")


if __name__ == "__main__":
    print("🚀 Starting BlueStock Mutual Fund ETL Ingestion Engine...\n" + "="*50)
    clean_nav_history()
    clean_transactions()
    clean_performance()
    print("="*50 + "\n All data cleaned and loaded into 'bluestock_mf.db' successfully!")

🚀 Starting BlueStock Mutual Fund ETL Ingestion Engine...
Reading from: C:\Users\dixit\bluestock_mf_capstone\data\raw\02_nav_history.csv
 Loaded fact_nav: 46000 rows

Reading from: C:\Users\dixit\bluestock_mf_capstone\data\raw\08_investor_transactions.csv
 Loaded fact_transactions: 32778 rows

Reading from: C:\Users\dixit\bluestock_mf_capstone\data\raw\07_scheme_performance.csv
 Loaded fact_aum: 40 rows
 Loaded fact_performance: 40 rows

 All data cleaned and loaded into 'bluestock_mf.db' successfully!
